# 03 — Model Development, Evaluation & Explainability

Loads preprocessed splits from notebook 02, trains/compares all models, evaluates on the test set, and runs the full SHAP/PDP explainability suite.

In [ ]:
import sys
sys.path.append('../src')
import joblib
from data_loader import load_config
from train_models import train_all_models, select_and_save_best_model
from evaluate import evaluate_all_models, plot_learning_curve
from explain import (compute_shap_values, plot_shap_summary, plot_shap_waterfall,
                      plot_shap_dependence, compute_permutation_importance,
                      plot_partial_dependence, plot_2d_interaction_pdp,
                      clinical_interpretation)

config = load_config('../config.yaml')
splits = joblib.load('../data/processed/splits.pkl')

## Train & compare all models (5-fold CV)

In [ ]:
fitted_models, leaderboard = train_all_models(splits, config)
leaderboard

## Select and persist best model

In [ ]:
best_path = select_and_save_best_model(fitted_models, leaderboard, config)
best_name = leaderboard.iloc[0]['model']
best_model = fitted_models[best_name]
best_name

## Test-set evaluation (all models)

In [ ]:
test_metrics = evaluate_all_models(fitted_models, splits, config)
test_metrics

## Learning curve for best model

In [ ]:
X_key = 'X_train_minmax' if best_name in ('linear_regression','elastic_net') else 'X_train_standard'
plot_learning_curve(best_model, splits[X_key], splits['y_train'], best_name, '../outputs/figures')

## SHAP global + local explanations

In [ ]:
n_sample = config['explainability']['shap_sample_size']
X_test_key = 'X_test_minmax' if best_name in ('linear_regression','elastic_net') else 'X_test_standard'
X_bg = splits[X_test_key].sample(min(100, len(splits[X_test_key])), random_state=42)
X_explain = splits[X_test_key].sample(min(n_sample, len(splits[X_test_key])), random_state=42)
shap_values = compute_shap_values(best_model, X_bg, X_explain)
plot_shap_summary(shap_values, X_explain, '../outputs/figures/shap_summary.png')

In [ ]:
plot_shap_waterfall(shap_values, 0, '../outputs/figures/shap_waterfall_patient0.png')

## Permutation importance & PDPs

In [ ]:
perm_imp = compute_permutation_importance(best_model, splits[X_test_key], splits['y_test'], config)
perm_imp.head(10)

In [ ]:
top5 = perm_imp['feature'].head(5).tolist()
plot_partial_dependence(best_model, splits[X_test_key], top5, '../outputs/figures/pdp_top5.png')

## Clinical interpretation of top features

In [ ]:
clinical_interpretation(top5)